In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
train_path = 'df_train.csv'  # Replace with your file path
test_path  = 'df_test.csv'   # Replace with your file path

df_train_raw = pd.read_csv(train_path, index_col=0)
df_test_raw  = pd.read_csv(test_path,  index_col=0)


In [3]:
def prepare_data(df):
        """Encode categorical columns and impute missing values with column medians."""
        df = df.copy()
        df = df.drop(columns=['time'], errors='ignore')
        df['Hawassa_wind_deg']  = df['Hawassa_wind_deg'].str.extract(r'(\d+)').astype(float)
        df['Kilimani_pressure'] = df['Kilimani_pressure'].str.extract(r'(\d+)').astype(float)
        df = df.fillna(df.median(numeric_only=True))
        return df

df_train = prepare_data(df_train_raw)
df_test  = prepare_data(df_test_raw)

print('Train shape:', df_train.shape)
print('Test shape: ', df_test.shape)

Train shape: (8763, 57)
Test shape:  (2920, 56)


### __Challenge 1: Variable Selection Using Correlation__

In Module 2 we found that five provincial temperature columns inflate VIF into the thousands because they measure the same underlying national weather pattern. Rather than feeding all 56 features into a regularized model blindly, we'll prune first: keep only the features that have a meaningful linear relationship with the target.

Features with low correlation to `energy_shortfall_3h` add noise without improving predictions — and they slow down every model evaluation unnecessarily.

### __Task__
Create a function named `select_features_by_correlation` that:
- Takes a DataFrame and a correlation threshold as input.
- Computes the absolute Pearson correlation of every numeric feature with `energy_shortfall_3h`.
- Prints the number of selected features.

In [4]:
def select_features_by_correlation(df, threshold=0.15):
    # Calculate absolute correlation with the target
    correlations = (
        df.corr(numeric_only=True)["energy_shortfall_3h"]
        .drop("energy_shortfall_3h")
        .abs()
    )

    # Select features meeting the threshold
    selected_features = correlations[
        correlations >= threshold
    ].index.tolist()

    # Print number of selected features
    print(f"Selected features: {len(selected_features)}")

    return selected_features

__Take Note__

```Python
.abs()
```

Because the challenge asks for absolute Pearson correlation. Therefore, both strong positive and strong negative relationships are selected.

In [5]:
selected_features = select_features_by_correlation(
    df_train,
    0.15
)

selected_features

Selected features: 22


['Sokoto_temp',
 'Sokoto_temp_min',
 'Sokoto_temp_max',
 'Sokoto_wind_speed',
 'Sokoto_pressure',
 'Kilimani_temp',
 'Kilimani_temp_min',
 'Kilimani_temp_max',
 'Kilimani_pressure',
 'Hawassa_temp',
 'Hawassa_temp_min',
 'Hawassa_temp_max',
 'Hawassa_wind_speed',
 'Hawassa_pressure',
 'Amanzi_temp',
 'Amanzi_temp_min',
 'Amanzi_temp_max',
 'Amanzi_pressure',
 'Akatsi_temp',
 'Akatsi_temp_min',
 'Akatsi_temp_max',
 'Akatsi_pressure']

We've reduced from 56 raw features to 22. The selected features are dominated by temperature and pressure across all five provinces — the variables most associated with heating/cooling demand and grid stability. `Sokoto_wind_speed` and `Hawassa_wind_speed` also pass the threshold, reflecting wind energy's contribution to the shortfall. Humidity and cloud cover fall just short of the 0.15 threshold this round. Every feature we trim is one fewer parameter the model has to estimate — and one fewer source of overfitting.

### __Challenge 2: Data Scaling and Ridge Regression__

The 22 selected features live on very different scales: temperatures in Kelvin (~285–305), wind speed in m/s (0–10), pressure in hPa (~995–1030). When regularization penalizes large coefficients, features on bigger scales get penalized more heavily — not because they matter less, but simply because their units are larger. **StandardScaler** removes this bias by transforming each feature to zero mean and unit variance.

**Ridge regression** (L2 regularization) then applies a penalty proportional to the *square* of each coefficient, shrinking large and unstable coefficients toward zero without eliminating any feature entirely. This directly addresses the multicollinearity we measured in Module 2.

**Critical rule:** The scaler must be **fit on training data only**, then applied (not refit) to the test data. Fitting on test data would leak future information into the model.

### __Task__
Create a function named `fit_ridge_model` that:
- Takes the training DataFrame, the list of selected features, and a regularization parameter `alpha` (default `1.0`).
- Splits the data 80-20 with `random_state=42`.
- Scales features using `StandardScaler` — fit on training, transform both.
- Fits a `Ridge` model on the scaled training set.
- Evaluates on the scaled test set.
- Returns `(r2, rmse, scaler, model)` — the scaler and model are returned for use in Challenges 4 and 5.

In [6]:
def fit_ridge_model(df, feature_cols, alpha=1.0):

    # Features and target
    X = df[feature_cols]
    y = df["energy_shortfall_3h"]
    
    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
    
    # Create scaler
    scaler = StandardScaler()
    
    # Fit ONLY on training data
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Transform test data using the fitted scaler
    X_test_scaled = scaler.transform(X_test)
    
    # Fit Ridge model
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate model
    r2 = r2_score(y_test, y_pred)
    
    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )
    
    return r2, rmse, scaler, model

In [7]:
r2, rmse, scaler, model = fit_ridge_model(
    df_train,
    selected_features
)

print(f"R²: {r2}")
print(f"RMSE: {rmse}")

R²: 0.1509892550454881
RMSE: 4581.03554336806


### __Challenge 3: LASSO Regression and Sparsity__

**LASSO regression** (L1 regularization) takes a fundamentally different approach. Rather than *shrinking* coefficients proportionally, LASSO applies a penalty proportional to the *absolute value* of each coefficient — which has the geometric effect of forcing small coefficients all the way to exactly zero. This produces **sparse models**: only the most important features survive.

LASSO is therefore doing two things simultaneously: regularization *and* automatic feature selection. At high enough `alpha`, it will eliminate features we chose by correlation thresholding, revealing which of our 17 are truly doing the heavy lifting.

### __Task__
Create a function named `fit_lasso_model` that:
- Takes the training DataFrame, the list of selected features, and a regularization parameter `alpha` (default `1.0`).
- Splits 80-20 with `random_state=42`, scales with `StandardScaler`.
- Fits a `Lasso` model on the scaled training set.
- Returns `(r2, rmse, n_nonzero)` where `n_nonzero` is the count of non-zero coefficients.

### __Expected Output__
```
LASSO R-squared: 0.12728536730988127
LASSO RMSE: 4957.636871918052
Non-zero coefficients: 18 out of 18
```

In [8]:
def fit_lasso_model(df, feature_cols, alpha=1.0):

    # Features and target
    X = df[feature_cols]
    y = df["energy_shortfall_3h"]
    
    # Split data 80-20
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
    
    # Scale features
    scaler = StandardScaler()
    
    # Fit scaler ONLY on training data
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Transform test data using the fitted scaler
    X_test_scaled = scaler.transform(X_test)
    
    # Fit LASSO model
    model = Lasso(
        alpha=alpha,
        max_iter=10000
    )
    
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    # Evaluation
    r2 = r2_score(y_test, y_pred)
    
    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )
    
    # Count non-zero coefficients
    n_nonzero = int(np.count_nonzero(model.coef_))
    
    return r2, rmse, n_nonzero

In [9]:
r2, rmse, n_nonzero = fit_lasso_model(
    df_train,
    selected_features
)

print(f"LASSO R-squared: {r2}")
print(f"LASSO RMSE: {rmse}")
print(
    f"Non-zero coefficients: "
    f"{n_nonzero} out of {len(selected_features)}"
)

LASSO R-squared: 0.15107120089580306
LASSO RMSE: 4580.814459056793
Non-zero coefficients: 22 out of 22


 ---
### __Challenge 4: Model Persistence with pickle__

We have a trained model that took seconds to fit — but in a production environment, models may take hours or days. More importantly, the Maji Ndogo Department of Energy's grid operators need to run this model at 3-hour intervals, 24 hours a day, without access to training data or a data scientist.

**Model persistence** solves this. Python's `pickle` module serializes any Python object — including trained scikit-learn models with all their fitted parameters — to a binary file on disk. Loading is instantaneous and produces an object that is byte-for-byte identical to the original.

### __Task__
Create a function named `save_and_reload_model` that:
- Takes a fitted model object and a file path as input.
- Saves the model to disk using `pickle.dump` (binary write mode: `'wb'`).
- Reloads it from disk using `pickle.load` (binary read mode: `'rb'`).
- Returns the reloaded model.

### __Expected Output__
```
Predictions match: True
```

In [10]:
import pickle

def save_and_reload_model(model, filepath):

    # Save the model
    with open(filepath, "wb") as file:
        pickle.dump(model, file)

    # Reload the model
    with open(filepath, "rb") as file:
        reloaded_model = pickle.load(file)

    return reloaded_model

In [11]:
reloaded_model = save_and_reload_model(
    model,
    "maji_ndogo_ridge_model.pkl"
)

In [12]:
r2, rmse, scaler, ridge_model = fit_ridge_model(df_train, selected_features)
reloaded_model = save_and_reload_model(ridge_model, 'maji_ndogo_ridge_model.pkl')
X_sample = df_train[selected_features].head(5)
# Verify the reloaded model produces identical predictions
X_scaled = scaler.transform(X_sample)
original_preds = ridge_model.predict(X_scaled)
reloaded_preds = reloaded_model.predict(X_scaled)
print('Original predictions:', original_preds)
print('Reloaded predictions:', reloaded_preds)
print('Predictions match:', np.allclose(original_preds, reloaded_preds))

Original predictions: [4059.6762807   988.14237644 4364.12735402 3398.6760809  4363.47544387]
Reloaded predictions: [4059.6762807   988.14237644 4364.12735402 3398.6760809  4363.47544387]
Predictions match: True


### __Challenge 5: Generating Predictions for Submission__

The final step: applying our trained Ridge model to the unseen test set and producing the submission file for the Maji Ndogo Department of Energy's leaderboard.

**Critical constraint:** The test set must be preprocessed using the **same scaler** that was fit on the training data — never refit it on the test data. Refitting would contaminate the model with information from the test set (data leakage), invalidating your evaluation.

### __Task__
Create a function named `generate_submission` that:
- Takes the raw test DataFrame (`df_test_raw`), the list of selected features, the fitted scaler, and the fitted Ridge model.
- Prepares the test data: encode categorical columns, impute missing values with medians.
- Scales the test features using the **already fitted** scaler (`.transform()` only — no `.fit_transform()`).
- Generates predictions using the Ridge model.
- Returns a DataFrame with columns `time` and `energy_shortfall_3h`.

### __Expected Output__
```
time  energy_shortfall_3h
0  2022-01-01 00:00:00          2085.070542
1  2022-01-01 03:00:00          4587.629794
2  2022-01-01 06:00:00          4963.214551
3  2022-01-01 09:00:00          5055.798913
4  2022-01-01 12:00:00          6843.955753
Submission shape: (2920, 2)
Saved to submission.csv
```

In [13]:
 def generate_submission(df_test_raw, feature_cols, scaler, model):

    # Make a copy so the raw test data is not modified
    df_test = df_test_raw.copy()

    # Encode categorical columns
    df_test = pd.get_dummies(df_test)

    # Make sure the test set has all selected features
    df_test = df_test.reindex(columns=feature_cols, fill_value=0)

    # Impute missing values using the test-set column medians
    # (Do not fit/refit the StandardScaler here)
    df_test = df_test.fillna(df_test.median())

    # Scale using the scaler fitted on the training data
    X_test_scaled = scaler.transform(df_test[feature_cols])

    # Generate predictions
    predictions = model.predict(X_test_scaled)

    # Create submission DataFrame
    submission = pd.DataFrame({
        "time": df_test_raw["time"],
        "energy_shortfall_3h": predictions
    })

    # Save submission
    submission.to_csv("submission.csv", index=False)

    return submission

In [14]:
# Input:
r2, rmse, scaler, ridge_model = fit_ridge_model(df_train, selected_features)
submission = generate_submission(df_test_raw, selected_features, scaler, ridge_model)

print(submission.head())
print(f'Submission shape: {submission.shape}')

submission.to_csv('submission.csv', index=False)

# Save to CSV for leaderboard submission
print('Saved to submission.csv')

                  time  energy_shortfall_3h
0  2022-01-01 00:00:00          2073.550875
1  2022-01-01 03:00:00          4564.590460
2  2022-01-01 06:00:00          4917.135883
3  2022-01-01 09:00:00          5048.119135
4  2022-01-01 12:00:00          6809.396751
Submission shape: (2920, 2)
Saved to submission.csv
